CONFIGURATION

In [0]:
volume_path = "/Volumes/document_ai_catalog/document_processing/document_files"

input_path = volume_path + "/input/"
output_path = volume_path + "/output/"
tracking_path = volume_path + "/tracking/"

category_tracking_file = tracking_path + "category_tracking.txt"
processed_files_file = tracking_path + "processed_files.txt"

model_name = "system.ai.gpt-oss-120b"

labels = {
    "Academic_Report": "Student records, academic performance, marks, attendance, examinations and educational reports.",
    "Employee_Information": "Employee records, staff details, employee performance, HR information and workforce reports.",
    "Sales_and_Finance": "Sales reports, revenue, expenses, financial statements, transactions and business accounts.",
    "Technical_Documentation": "Technical specifications, software documentation, system reports and engineering documents.",
    "Other": "Documents that do not fit into any of the above categories."
}

import json
import re

labels_json = json.dumps(labels)

print("Pipeline configuration completed.")

Pipeline configuration completed.


HELPER FUNCTIONS

In [0]:
def read_file(path):

    try:
        return dbutils.fs.head(path, 10 * 1024 * 1024)

    except Exception:
        return ""


def append_to_file(path, new_content):

    existing_content = read_file(path)

    if existing_content.strip():
        final_content = existing_content.rstrip() + "\n" + new_content
    else:
        final_content = new_content

    dbutils.fs.put(
        path,
        final_content + "\n",
        overwrite=True
    )


def clean_schema_names(node):

    if isinstance(node, dict):

        node.pop("required", None)

        if "properties" in node:

            updated_properties = {}

            for name, value in node["properties"].items():

                new_name = re.sub(r"[^\w.-]", "_", name)
                new_name = new_name.lstrip("_")

                if not new_name:
                    raise ValueError(f"Invalid column name: {name}")

                updated_properties[new_name] = clean_schema_names(value)

            node["properties"] = updated_properties

        for key in node:
            if key != "properties":
                node[key] = clean_schema_names(node[key])

    elif isinstance(node, list):

        for i in range(len(node)):
            node[i] = clean_schema_names(node[i])

    return node


def save_csv(df, file_name):

    final_file = output_path + file_name
    temp_folder = output_path + "temp_" + file_name.replace(".csv", "")

    dbutils.fs.rm(temp_folder, True)
    dbutils.fs.rm(final_file, True)

    df.coalesce(1).write \
        .option("header", True) \
        .mode("overwrite") \
        .csv(temp_folder)

    csv_file = [
        file.path for file in dbutils.fs.ls(temp_folder)
        if file.name.endswith(".csv")
    ][0]

    dbutils.fs.mv(csv_file, final_file)

    dbutils.fs.rm(temp_folder, True)

    print("Saved:", file_name)

PARSING THE DOCUMENT AND IDENTIFYING THE TABLES

In [0]:
def parse_document(file_path):

    binary_df = spark.read.format("binaryFile").load(file_path)

    binary_df.createOrReplaceTempView("binary_documents")

    parsed_df = spark.sql("""
        SELECT
            path,
            ai_parse_document(
                content,
                MAP('version', '2.0')
            ) AS parsed
        FROM binary_documents
    """)

    parsed_data = parsed_df.first()["parsed"].toPython()

    if parsed_data.get("error_status") is not None:
        raise Exception(str(parsed_data["error_status"]))

    parsed_df.createOrReplaceTempView("parsed_documents")

    tables_df = spark.sql("""
        SELECT
            element.pos AS table_position,
            element.value:content::STRING AS table_html,
            element.value:confidence::DOUBLE AS confidence
        FROM parsed_documents,
        LATERAL variant_explode(
            parsed:document:elements
        ) AS element
        WHERE element.value:type::STRING = 'table'
        ORDER BY table_position
    """)

    return tables_df

GENERATING A SCHEMA FOR EACH TABLE

In [0]:
def generate_schema(table_html):

    prompt = f"""
    Analyze this HTML table.

    Generate a JSON Schema for extracting all its data rows.

    Use table headers as field names.
    Infer suitable data types from the headers and sample values.
    Keep identifiers such as register numbers as strings.
    Do not invent columns.
    Do not include required fields.
    Use only string, integer, number or boolean.

    Return only valid JSON without Markdown.

    Use this structure:

    {{
        "type": "object",
        "properties": {{
            "rows": {{
                "type": "array",
                "items": {{
                    "type": "object",
                    "properties": {{}}
                }}
            }}
        }}
    }}

    HTML TABLE:
    {table_html}
    """

    prompt = prompt.replace("'", "''")

    schema_result = spark.sql(f"""
        SELECT ai_query(
            '{model_name}',
            '{prompt}'
        ) AS generated_schema
    """).first()["generated_schema"]

    schema_result = schema_result.replace("```json", "").replace("```", "").strip()

    schema = json.loads(schema_result)

    schema = clean_schema_names(schema)

    # ai_extract expects the top-level properties, not the wrapper.
    return json.dumps(schema["properties"])

EXTRACTING TABLE DATA

In [0]:
def extract_table(table_html, generated_schema):

    table_html = table_html.replace("'", "''")
    generated_schema = generated_schema.replace("'", "''")

    result_df = spark.sql(f"""
        SELECT ai_extract(
            '{table_html}',
            '{generated_schema}',
            MAP('version', '2.1')
        ) AS extracted
    """)

    extracted_data = result_df.first()["extracted"].toPython()

    if extracted_data["error_message"] is not None:
        raise Exception(extracted_data["error_message"])

    rows = extracted_data["response"]["rows"]

    clean_rows = []

    for row in rows:

        clean_row = {}

        for column, value in row.items():
            clean_row[column] = value["value"]

        clean_rows.append(clean_row)

    if not clean_rows:
        raise ValueError("No data rows extracted from the table.")

    return spark.createDataFrame(clean_rows)

CLASSIFYING ALL TABLES IN ONE SQL QUERY

In [0]:
def classify_tables(table_records):

    labels_sql = labels_json.replace("'", "''")

    queries = []

    for table in table_records:

        position = table["table_position"]
        table_html = table["table_html"].replace("'", "''")

        queries.append(f"""
            SELECT
                {position} AS table_position,
                ai_classify(
                    '{table_html}',
                    '{labels_sql}',
                    MAP(
                        'version', '2.1',
                        'enableConfidenceScores', 'true',
                        'enableRationales', 'true'
                    )
                ) AS classification
        """)

    classification_df = spark.sql(
        " UNION ALL ".join(queries)
    )

    classification_results = []

    for row in classification_df.collect():

        data = row["classification"].toPython()

        if data["error_message"] is not None:
            raise Exception(data["error_message"])

        result = data["response"][0]

        classification_results.append({
            "Table_Position": row["table_position"],
            "Category": result["value"]
        })

    return classification_results

PROCESSING THE COMPLETE DOCUMENT

In [0]:
def process_document(file_path):

    document_name = file_path.split("/")[-1].rsplit(".", 1)[0]

    print("Processing:", document_name)
    tables_df = parse_document(file_path)

    table_records = tables_df.collect()

    if not table_records:
        raise ValueError("No tables found in this document.")

    print("Tables identified:", len(table_records))

    extracted_tables = {}

    for table in table_records:

        position = table["table_position"]

        generated_schema = generate_schema(table["table_html"])

        final_df = extract_table(
            table["table_html"],
            generated_schema
        )

        extracted_tables[position] = final_df

        print("Extracted table:", position)

    classifications = classify_tables(table_records)

    category_mapping = {
        item["Table_Position"]: item["Category"]
        for item in classifications
    }
    tracking_entries = []

    for position, df in extracted_tables.items():

        file_name = f"{document_name}_table_{position}.csv"

        save_csv(df, file_name)

        category = category_mapping[position]

        tracking_entries.append(
            f"{file_name} : {category}"
        )
    append_to_file(
        category_tracking_file,
        "\n".join(tracking_entries)
    )

    print("Category tracking updated.")

    print("Successfully processed:", document_name)

DISCOVERING AND PROCESSING THE UNPROCESSED DOCUMENTS

In [0]:

processed_content = read_file(processed_files_file)

processed_keys = set(processed_content.splitlines())

input_files = dbutils.fs.ls(input_path)

docx_files = [
    file for file in input_files
    if file.name.lower().endswith(".docx")
]

print("Word documents found:", len(docx_files))

for file in docx_files:

    file_key = (
        f"{file.path}|{file.size}|{file.modificationTime}"
    )

    if file_key in processed_keys:

        print("Already processed:", file.name)
        continue

    print("\nStarting:", file.name)

    process_document(file.path)

    append_to_file(
        processed_files_file,
        file_key
    )

    processed_keys.add(file_key)

    print("Completed:", file.name)

print("\nDocument processing workflow completed.")

Word documents found: 1

Starting: student_academic_report.docx
Processing: student_academic_report
Tables identified: 2
Extracted table: 3
Extracted table: 5
Saved: student_academic_report_table_3.csv
Saved: student_academic_report_table_5.csv
Wrote 108 bytes.
Category tracking updated.
Successfully processed: student_academic_report
Wrote 124 bytes.
Completed: student_academic_report.docx

Document processing workflow completed.


VERIFICATION

In [0]:
print("OUTPUT FILES")
display(dbutils.fs.ls(output_path))
print("CATEGORY TRACKING")
print(read_file(category_tracking_file))
print("PROCESSED DOCUMENTS")
print(read_file(processed_files_file))

OUTPUT FILES


path,name,size,modificationTime
dbfs:/Volumes/document_ai_catalog/document_processing/document_files/output/student_academic_report_table_3.csv,student_academic_report_table_3.csv,254,1795763727000
dbfs:/Volumes/document_ai_catalog/document_processing/document_files/output/student_academic_report_table_5.csv,student_academic_report_table_5.csv,291,1795763730000


CATEGORY TRACKING
student_academic_report_table_3.csv : Academic_Report
student_academic_report_table_5.csv : Academic_Report

PROCESSED DOCUMENTS
dbfs:/Volumes/document_ai_catalog/document_processing/document_files/input/student_academic_report.docx|37419|1795763588000

